# 🎬 London Beer Flood 1814 - GPU Cloud Render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunmalpani/london-beer-flood-1814/blob/main/render_colab.ipynb)

### ⚡ Quick Start:
1. **Enable GPU**: Click **Runtime** &rarr; **Change runtime type** &rarr; select **T4 GPU** &rarr; **Save**.
2. **Run Render**: Click **Runtime** &rarr; **Run all** (or press `Ctrl+F9`).
3. **Done**: The video renders in ~2-3 minutes with full hardware acceleration, muxes audio, and automatically downloads `final_beer_flood_1814.mp4` to your computer!

In [ ]:
# Step 1: Check NVIDIA GPU
!nvidia-smi

In [ ]:
# Step 2: Clone repository & install dependencies
import os
if not os.path.exists('london-beer-flood-1814'):
    !git clone https://github.com/arjunmalpani/london-beer-flood-1814.git
%cd london-beer-flood-1814
!git pull origin main
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs ffmpeg xvfb libegl1-mesa-dev libgl1-mesa-dev
!npm install --legacy-peer-deps

In [ ]:
# Step 3: Test GPU WebGL rendering with frame 100
!mkdir -p out
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion still src/index.ts BeerFlood out/test_gpu.jpg \
    --frame=100 \
    --gl=angle-egl \
    --timeout=60000 \
    --overwrite
!ls -lh out/test_gpu.jpg

In [ ]:
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion render src/index.ts BeerFlood out/raw_video.mp4 \
    --chrome-mode="chrome-for-testing" \
    --gl=angle-egl \
    --hardware-acceleration=if-possible \
    --scale=0.6667 \
    --concurrency=4 \
    --timeout=120000 \
    --overwrite

In [ ]:
# Step 5: Mux master audio mix (-15 LUFS stereo) & verify
!ffmpeg -y -i out/raw_video.mp4 -i public/audio/mix.wav \
    -map 0:v:0 -map 1:a:0 \
    -c:v copy \
    -c:a aac -b:a 320k \
    -shortest \
    out/final_beer_flood_1814.mp4

!ls -lh out/final_beer_flood_1814.mp4
print("\n Render and Master Audio Muxing Completed Successfully!")

In [ ]:
# Step 6: Download the finished video
from google.colab import files
files.download('out/final_beer_flood_1814.mp4')